In [ ]:
# Python dictionary representation of the knowledge base
knowledge_base = {
    "facts": {
        "gpa_categories": [
            "GPA Above 3.5",
            "GPA Between 3.0 and 3.49",
            "GPA Below 3.0"
        ],
        "attendance_categories": [
            "Attendance Above 80%",
            "Attendance Below 80%"
        ],
        "discipline": [
            "No Disciplinary Cases",
            "Has Disciplinary Cases"
        ],
        "fees": [
            "No Outstanding Fees",
            "Has Outstanding Fees"
        ],
        "courses": [
            "Completed Prerequisite Courses",
            "Missing Prerequisite Courses"
        ]
    },
    "rules": {
        "Scholarship": {
            "if": ["GPA Above 3.5", "Attendance Above 80%", "No Disciplinary Cases"],
            "then": "Eligible for Scholarship"
        },
        "DeanList": {
            "if": ["GPA Above 3.5", "Attendance Above 80%"],
            "then": "Dean's List Candidate"
        },
        "Graduation": {
            "if": ["GPA Above 3.0", "Completed Prerequisite Courses", "No Outstanding Fees"],
            "then": "Eligible for Graduation"
        },
        "Probation": {
            "if": ["GPA Below 3.0"],
            "then": "Academic Probation"
        },
        "RegistrationBlocked": {
            "if": ["Has Outstanding Fees"],
            "then": "Registration Blocked"
        }
    },
    "conclusions": [
        "Eligible for Scholarship",
        "Dean's List Candidate",
        "Eligible for Graduation",
        "Academic Probation",
        "Registration Blocked"
    ]
}


In [1]:


import tkinter as tk
from tkinter import ttk, messagebox

KB = {
    "rules": [
        {"name": "Scholarship",
         "if": ["GPA Above 3.5", "Attendance Above 80%", "No Disciplinary Cases"],
         "then": "Eligible for Scholarship"},
        {"name": "Dean's List",
         "if": ["GPA Above 3.5", "Attendance Above 80%"],
         "then": "Dean's List Candidate"},
        {"name": "Graduation",
         "if": ["GPA Above 3.0", "Completed Prerequisite Courses", "No Outstanding Fees"],
         "then": "Eligible for Graduation"},
        {"name": "Probation",
         "if": ["GPA Below 3.0"],
         "then": "Academic Probation"},
        {"name": "Registration Block",
         "if": ["Has Outstanding Fees"],
         "then": "Registration Blocked"}
    ]
}


def map_student_to_facts(student):
    facts = set()
    gpa = student.get("gpa")
    if gpa is None:
        raise ValueError("Missing 'gpa'")
    if gpa > 3.5:
        facts.add("GPA Above 3.5"); facts.add("GPA Above 3.0")
    elif 3.0 < gpa <= 3.5:
        facts.add("GPA Between 3.0 and 3.49"); facts.add("GPA Above 3.0")
    elif gpa == 3.0:
        facts.add("GPA Above 3.0")
    else:
        facts.add("GPA Below 3.0")

    attendance = student.get("attendance")
    if attendance is None:
        raise ValueError("Missing 'attendance'")
    if attendance > 80:
        facts.add("Attendance Above 80%")
    else:
        facts.add("Attendance Below 80%")

    if student.get("has_disciplinary_case"):
        facts.add("Has Disciplinary Cases")
    else:
        facts.add("No Disciplinary Cases")

    if student.get("has_outstanding_fees"):
        facts.add("Has Outstanding Fees")
    else:
        facts.add("No Outstanding Fees")

    if student.get("completed_prereqs"):
        facts.add("Completed Prerequisite Courses")
    else:
        facts.add("Missing Prerequisite Courses")

    return facts


def evaluate_student(student):
    facts = map_student_to_facts(student)
    conclusions = set()
    fired_rules = []
    for rule in KB["rules"]:
        antecedents = set(rule["if"])
        if antecedents.issubset(facts):
            conclusions.add(rule["then"])
            fired_rules.append(rule["name"])
    return {"facts": facts, "conclusions": conclusions, "fired_rules": fired_rules}


def build_gui():
    root = tk.Tk()
    root.title("Academic Advisor")

    frm = ttk.Frame(root, padding=12)
    frm.grid(row=0, column=0, sticky="nsew")

    ttk.Label(frm, text="Student name:").grid(row=0, column=0, sticky="w")
    name_var = tk.StringVar()
    ttk.Entry(frm, textvariable=name_var, width=30).grid(row=0, column=1, sticky="w")

    ttk.Label(frm, text="GPA (0.0 - 4.0):").grid(row=1, column=0, sticky="w")
    gpa_var = tk.StringVar()
    ttk.Entry(frm, textvariable=gpa_var, width=10).grid(row=1, column=1, sticky="w")

    ttk.Label(frm, text="Attendance % (0 - 100):").grid(row=2, column=0, sticky="w")
    att_var = tk.StringVar()
    ttk.Entry(frm, textvariable=att_var, width=10).grid(row=2, column=1, sticky="w")

    disc_var = tk.BooleanVar(value=False)
    fees_var = tk.BooleanVar(value=False)
    prereq_var = tk.BooleanVar(value=False)

    ttk.Checkbutton(frm, text="Has disciplinary case", variable=disc_var).grid(row=3, column=0, columnspan=2, sticky="w")
    ttk.Checkbutton(frm, text="Has outstanding fees", variable=fees_var).grid(row=4, column=0, columnspan=2, sticky="w")
    ttk.Checkbutton(frm, text="Completed prerequisite courses", variable=prereq_var).grid(row=5, column=0, columnspan=2, sticky="w")

    output = tk.Text(frm, width=60, height=12, wrap="word")
    output.grid(row=7, column=0, columnspan=2, pady=(8, 0))

    def on_evaluate():
        try:
            gpa = float(gpa_var.get())
            attendance = float(att_var.get())
            if not (0.0 <= gpa <= 4.0 and 0.0 <= attendance <= 100.0):
                raise ValueError
        except Exception:
            messagebox.showerror("Input error", "Please enter valid numeric GPA (0-4) and attendance (0-100).")
            return
        student = {
            "gpa": gpa,
            "attendance": attendance,
            "has_disciplinary_case": disc_var.get(),
            "has_outstanding_fees": fees_var.get(),
            "completed_prereqs": prereq_var.get()
        }
        res = evaluate_student(student)
        output.delete("1.0", tk.END)
        if name_var.get().strip():
            output.insert(tk.END, f"Student: {name_var.get().strip()}\n")
        output.insert(tk.END, "Normalized facts: " + ", ".join(sorted(res["facts"])) + "\n\n")
        if res["conclusions"]:
            output.insert(tk.END, "Conclusions:\n")
            for c in sorted(res["conclusions"]):
                output.insert(tk.END, f" - {c}\n")
        else:
            output.insert(tk.END, "Conclusions: None\n")
        if res["fired_rules"]:
            output.insert(tk.END, "\nRules fired: " + ", ".join(res["fired_rules"]))
        else:
            output.insert(tk.END, "\nNo rules fired.")

    ttk.Button(frm, text="Evaluate", command=on_evaluate).grid(row=6, column=0, columnspan=2, pady=(6, 0))

    root.mainloop()


if __name__ == "__main__":
    build_gui()
